In [112]:
import yaml
import re
import os

macro = {}
total_per_energy = {}

dnn_layers = {
    "alexnet": 8,
    "vgg16": 16,
    "dpt_large": 227,
    "gpt2_medium": 145,
    "mobilebert": 409,
    "vision_transformer": 98,
    "resnet18": 21,
    "mobilenet_v3": 64,
    "alexnet-1layer": 1
}

# Define the architecture
array_sizes = 128
tech_node = 32
time = "fast"

dnn_list = ["alexnet", "vgg16", "resnet18"]
for DNN in dnn_list:
    
    dnn_layer = dnn_layers.get(DNN, None)
    
    def read_and_compute_multiplication(filename, i, format_layer):
        filename = f"/home/workspace/example_designs/example_designs/simple_weight_stationary/{filename}/{filename}_{array_sizes}/outputs/{i:0{format_layer}d}/timeloop-mapper.map.txt"
        with open(filename, 'r') as file:
            lines = file.readlines()
    
        start_searching = False
        ranges = []
    
        for line in lines:
            # Start searching when 'inter_macro_in_system_spatial' is found
            if "inter_PE_spatial" in line:
                start_searching = True
            
            # Stop searching if we leave the 'inter_macro_in_system_spatial' section
            elif start_searching and "inter_" in line and "inter_PE_spatial" not in line:
                break
    
            # If we are in the 'inter_macro_in_system_spatial' section, look for the ranges
            if start_searching:
                if "for" in line:
                    range_size = extract_range(line)
                    if range_size is not None:
                        ranges.append(range_size)
                elif line.strip() == "":  # Exit the section on an empty line
                    break
    
        # Compute the multiplication of all ranges, or return 1 if no ranges were found
        if not ranges:
            print(f"All macro: 1")
            return 1
    
        result = 1
        for size in ranges:
            result *= size
            print(f"Size: {size}")
        print(f"All macro: {result}")
        return result
    
    
    def extract_layer_number(folder_name):
        m = re.match(r'^(\d+)$', folder_name)
        return int(m.group(1)) if m else float('inf')
    
    
    def extract_outputs(model, format_layer):
        
        output_dir = "/home/workspace/example_designs/example_designs/simple_weight_stationary/" + model +  "/" + model + "_" + str(array_sizes) +"/outputs/" 
        print(output_dir)
        #for model in models:    
        throughput = 0   
        energy = 0
        area = 0
        cycles = 0
        folders = sorted(
        (f for f in os.listdir(output_dir)
           if os.path.isdir(os.path.join(output_dir, f)) and f.isdigit()),
        key=extract_layer_number)
    
        print(folders)
        area_per_many_macro = 0
        area_layer = 0
        j = 0
        total_sum = 0
        total_adc_area = 0  # Total area for ADC across all layers
        total_other_areas = {}  # Dictionary to store total areas for other components
        total_col_area = 0
        total_row_area = 0
        total_cim_unit_area = 0
        total_area = 0
        total_macro = 0
        total_inter_PE           = 0.0
        total_weight_reg         = 0.0
        total_input_activation   = 0.0
        total_output_activation  = 0.0
        total_shared_glb         = 0.0
        total_pe_spad            = 0.0
        total_mac                = 0.0
        
        for folder in folders:
            total = 0
            print(f"Processing layer: {folder}")
            # print(folder)
            if os.path.isdir(output_dir + folder):
                stat_file = output_dir + folder + "/timeloop-mapper.stats.txt"
                art_file = output_dir + folder + "/timeloop-mapper.ART.yaml"
        
                with open(stat_file, 'r') as f:
                    energy_stat = [line for line in f if line.startswith("Energy:")][0]
                    energy_uj = energy_stat.split(' ')[1] # uJ
                    energy += float(energy_uj)
                    # print(f"Energy for this layer: {float(energy_uj)} uj")
                
                with open(stat_file, 'r') as f:
                    cycle_stat = [line for line in f if line.startswith("Cycles:")][0]                  
                    cycle = cycle_stat.split(' ')[1] 
                    cycles += float(cycle)
                    print(f"Cycle: {cycle}")
                print(f"Total Cycle: {cycles}")
                    
            if j < dnn_layer:
                with open(art_file, 'r') as file:
                    data = yaml.safe_load(file)
                
                    # Extract and print the name, area, number of entries, and ratio for non-zero areas
                for table in data['ART']['tables']:
                    name = table.get('name')
                    area = table.get('area')
                    if area != 0:
                        # Simplify the name for the output
                        if 'system_top_level.' in name:
                            name = name.split('system_top_level.')[-1]
                        
                        # Extract the number of entries from the name
                        if '[' in name:
                            name_part, entries_part = name.split('[')
                            number_of_entries = entries_part.split('..')[1].replace(']', '')
                            name = name_part.strip()
                        else:
                            number_of_entries = 'Unknown'
                        
                        macro = read_and_compute_multiplication(model, j, format_layer)
                        # Calculate total ratio
                        if number_of_entries != 'Unknown' and number_of_entries.isdigit():
                            total_ratio = (area / 4096) * macro * int(number_of_entries)
                        else:
                            total_ratio = 0
                        
                        total += total_ratio
                        
                        # Accumulate area for ADC or other components
                        if 'adc' in name.lower():
                            total_adc_area += total_ratio
                        else:
                            if name not in total_other_areas:
                                total_other_areas[name] = 0
                            total_other_areas[name] += total_ratio
            
                        print(f"{name.replace('_', ' ')} area: {area}, number of {name}: {number_of_entries}, \ntotal {name.replace('_', ' ')} area: ({area}*{number_of_entries}/4096) * macro = {total_ratio}\n")
                j += 1 
                total_sum += total
                total_macro += macro
                print(f"Total Area for layer {j}: {total}\n")
                print(f"Total Macro for layer {j}: {total_macro}\n")
            print(f"Total Area for all layers: {total_sum}")
            print(f"Total Area for ADC across all layers: {total_adc_area}")
        
        # Print total area for other components
        for component, area in total_other_areas.items():
            print(f"Total Area for {component.replace('_', ' ')} across all layers: {area}")
            # print(component.replace('_', ' '))
            if component.replace('_', ' ') ==  "column drivers":
                total_col_area = area
            elif component.replace('_', ' ') ==  "row drivers":
                total_row_area = area
    
            elif component.replace('_', ' ') == "inter PE spatial":
                total_inter_PE = area
        
            elif component.replace('_', ' ') == "weight reg":
                total_weight_reg = area
        
            elif component.replace('_', ' ') == "input activation reg":
                total_input_activation = area
        
            elif component.replace('_', ' ') == "output activation reg":
                total_output_activation = area
        
            elif component.replace('_', ' ') == "shared glb":
                total_shared_glb = area
        
            elif component.replace('_', ' ') == "pe spad":
                total_pe_spad = area
        
            elif component.replace('_', ' ') == "mac":
                total_mac = area
        
            else:
                # anything else you didn’t explicitly match
                total_cim_unit_area += area
                                    
        energy *= 1e-6 # conver to J
        total_area = total_sum * 1e-6 # mm2
        throughput = 1 / (cycles * 1e-7) # s
       # convert all areas from µm² to mm²
    
        total_adc_area = total_adc_area * 1e-6 # mm2
        total_col_area = total_col_area * 1e-6 # mm2
        total_row_area = total_row_area * 1e-6 # mm2
        total_cim_unit_area = total_cim_unit_area * 1e-6 # mm2
        total_inter_PE           = total_inter_PE          * 1e-6
        total_weight_reg         = total_weight_reg        * 1e-6
        total_input_activation   = total_input_activation  * 1e-6
        total_output_activation  = total_output_activation * 1e-6
        total_shared_glb         = total_shared_glb        * 1e-6
        total_pe_spad            = total_pe_spad           * 1e-6
        total_mac                = total_mac               * 1e-6
    
        print("\n")
        print(f"Total Area for ADC across all layers: {total_adc_area}")
        print(f"Total Area for column driver across all layers: {total_col_area}")
        print(f"Total Area for row driver across all layers: {total_row_area}")
        print(f"Total Area for cim unit across all layers: {total_cim_unit_area}")
        print(f"Total Area for inter PE spatial across all layers: {total_inter_PE}")
        print(f"Total Area for weight register across all layers: {total_weight_reg}")
        print(f"Total Area for input activation register across all layers: {total_input_activation}")
        print(f"Total Area for output activation register across all layers: {total_output_activation}")
        print(f"Total Area for shared global buffer across all layers: {total_shared_glb}")
        print(f"Total Area for PE scratchpad across all layers: {total_pe_spad}")
        print(f"Total Area for MAC units across all layers: {total_mac}")
        print(f"Total Macro across all layers: {total_macro}")
        print ("Model \t Energy/inference (J) \t Area (mm2) \t Throughput (inf/s)")
        print ("{} \t {:.2e} \t {:.2f} \t {:.2e}".format(model, energy, total_area, throughput))
        
        return total_area, total_adc_area, total_col_area, total_row_area, total_cim_unit_area, total_macro, energy, throughput


In [113]:
# Initialize lists for each DNN
num_dnns = len(dnn_list)
total_area = [0] * (2 * num_dnns)
total_adc_area = [0] * (2 * num_dnns)
total_col_area = [0] * (2 * num_dnns)
total_row_area = [0] * (2 * num_dnns)
total_cim_unit_area = [0] * (2 * num_dnns)
total_macro = [0] * (2 * num_dnns)
energy = [0] * (2 * num_dnns)
throughput = [0] * (2 * num_dnns)

# Loop through both DNNs and extract their outputs for SRAM and RRAM
for i, dnn in enumerate(dnn_list):
    dnn_layer = dnn_layers.get(dnn, None)
    if dnn_layer < 10:
        format_layer = 1
    elif dnn_layer >= 10 and dnn_layer < 100:
        format_layer = 2
    else:
        format_layer = 3
    total_area[2 * i], total_adc_area[2 * i], total_col_area[2 * i], total_row_area[2 * i], total_cim_unit_area[2 * i], total_macro[2 * i], energy[2 * i], throughput[2 * i] = extract_outputs(f"{dnn}", format_layer)

/home/workspace/example_designs/example_designs/simple_weight_stationary/alexnet/alexnet_128/outputs/
['0', '1', '2', '3', '4', '5', '6', '7']
Processing layer: 0
Cycle: 1585176

Total Cycle: 1585176.0
Size: 32
Size: 3
All macro: 96
inter PE spatial area: 1.0, number of inter_PE_spatial: 1, 
total inter PE spatial area: (1.0*1/4096) * macro = 0.0234375

Size: 32
Size: 3
All macro: 96
weight reg area: 68.9755, number of weight_reg: 16384, 
total weight reg area: (68.9755*16384/4096) * macro = 26486.591999999997

Size: 32
Size: 3
All macro: 96
input activation reg area: 68.9755, number of input_activation_reg: 16384, 
total input activation reg area: (68.9755*16384/4096) * macro = 26486.591999999997

Size: 32
Size: 3
All macro: 96
output activation reg area: 68.9755, number of output_activation_reg: 16384, 
total output activation reg area: (68.9755*16384/4096) * macro = 26486.591999999997

Size: 32
Size: 3
All macro: 96
shared glb area: 118878.0, number of shared_glb: 1, 
total shared g

In [115]:
print("\n### Area and Performance Results ###\n")

for i, dnn in enumerate(dnn_list):
    print(f"Results for {dnn}:")
    # print(f"  Total Area: {total_area[2 * i]} mm2")
    # print(f"  Total Macro: {total_macro[2 * i]}")
    print(f"  Energy: {energy[2 * i]} J")
    print(f"  Throughput: {throughput[2 * i]} inf/s \n")



### Area and Performance Results ###

Results for alexnet:
  Energy: 0.009168939999999999 J
  Throughput: 1.1834571624010926 inf/s 

Results for vgg16:
  Energy: 0.05647872 J
  Throughput: 0.05276459460244369 inf/s 

Results for resnet18:
  Energy: 0.005277200000000001 J
  Throughput: 0.4516639412511678 inf/s 

